# ch02 练习：数据读写

> 共 7 道基础题 + 1 道综合题（拆 3 小步，合计 10 个空）
> 建议限时：**25 分钟**

**所有题目都基于 `data/` 下的真实文件**，读不通就是真的读不通。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 130)

DATA = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

print("数据目录:", sorted(p.name for p in DATA.iterdir()))

---

### TODO(1) 读 GBK 编码的文件

`device_defects_gbk.csv` 是 GBK 编码。请把它读进来，变量名 `gbk`。

先试一下不加 `encoding` 参数会怎样，再加参数。

In [ ]:
gbk = pd.read_csv(DATA / "device_defects_gbk.csv", encoding="gbk")

# ---- 验收 ----
assert isinstance(gbk, pd.DataFrame), "gbk 应该是 DataFrame"
assert gbk.shape == (270, 13), f"形状应为 (270, 13)，实际 {gbk.shape}"
assert "缺陷类型" in gbk.columns, "列名乱码了，说明编码不对"
assert gbk["台区编号"].iloc[0].startswith("STATION_"), "内容乱码"
print("TODO(1) 通过 ->", gbk.shape)

### TODO(2) 用 `dtype` 锁住类型

读 `device_defects_raw.csv`，要求 **所有列都读成字符串**（`dtype=str`），
变量名 `raw_str`。

这样做的意义：原始文件里的 `负荷值` 混着单位、千分位、缺失占位符，
交给 pandas 猜类型只会得到一堆意外。

In [ ]:
raw_str = pd.read_csv(DATA / "device_defects_raw.csv", dtype=str)

# ---- 验收 ----
assert raw_str.shape == (270, 13), f"形状应为 (270, 13)，实际 {raw_str.shape}"
assert all(str(dt) == "str" for dt in raw_str.dtypes), "还有列不是字符串类型"
assert raw_str["负荷值"].str.contains("kW", na=False).sum() == 5, "带单位的行数不对，检查是不是被转成数值了"
print("TODO(2) 通过 -> 全部列 dtype 均为 str")

### TODO(3) `usecols` + `nrows` 快速摸底

只想看 `记录ID`、`台区编号`、`缺陷类型` 三列的**前 8 行**，变量名 `peek`。

要求用 `usecols` 和 `nrows` **在读取阶段**就裁掉多余数据。

In [ ]:
peek = pd.read_csv(DATA / "device_defects.csv", usecols=["记录ID", "台区编号", "缺陷类型"], nrows=8)

# ---- 验收 ----
assert peek.shape == (8, 3), f"形状应为 (8, 3)，实际 {peek.shape}"
assert list(peek.columns) == ["记录ID", "台区编号", "缺陷类型"], "列不对"
print("TODO(3) 通过 ->\n", peek.head(3))

### TODO(4) `na_values` 统一缺失写法

`device_defects_raw.csv` 的 `负荷值` 列里有 **3 种**缺失写法混用：`N/A`、`—`、空串。
请只用一次 `read_csv`，把它们全部识别为缺失值，变量名 `normalized`。

读完后 `负荷值` 列的缺失个数应该是 **21**。

In [ ]:
normalized = pd.read_csv(
    DATA / "device_defects_raw.csv",
    dtype=str,
    na_values=["N/A", "NA", "—", "-", "", "无", "/"],
)

# ---- 验收 ----
assert int(normalized["负荷值"].isna().sum()) == 21, (
    f"缺失个数应为 21，实际 {int(normalized['负荷值'].isna().sum())}，有缺失写法漏网了"
)
assert int((normalized["负荷值"] == "—").sum()) == 0, "还有 '—' 残留在值里"
print("TODO(4) 通过 -> 负荷值缺失 =", int(normalized["负荷值"].isna().sum()))

### TODO(5) `parse_dates` + `index_col`

读 `load_curve.csv`，要求：
- `时间戳` 列**解析为 datetime**
- `时间戳` 同时作为 **索引**
- 结果存入 `ts`

读完后 `ts.index` 应该是 `DatetimeIndex`。

In [ ]:
ts = pd.read_csv(DATA / "load_curve.csv", parse_dates=["时间戳"], index_col="时间戳")

# ---- 验收 ----
assert isinstance(ts.index, pd.DatetimeIndex), f"索引类型不对：{type(ts.index).__name__}"
assert ts.shape == (6480, 4), f"形状应为 (6480, 4)，实际 {ts.shape}（原 5 列，时间列已变索引）"
assert ts.index.nunique() == 2160, f"不同时刻数应为 2160（90 天 x 24 小时），实际 {ts.index.nunique()}"
print("TODO(5) 通过 ->", ts.shape, "| 时间范围:", ts.index.min(), "~", ts.index.max())
print("  注意：现在是 3 个台区堆叠，索引不是单调递增的，做 resample 前要先 sort_index")

### TODO(6) `chunksize` 分块聚合

用 `chunksize=100` 分块读 `device_defects.csv`，
在不把整个文件读进内存的前提下统计出：
- `n_chunks`：一共几块
- `row_count`：总行数

## 坑位提示

`pd.read_csv(..., chunksize=...)` **不直接返回 DataFrame**，而是一个可迭代的读取器。

In [ ]:
n_chunks = 0
row_count = 0

reader = pd.read_csv(DATA / "device_defects.csv", chunksize=100)

for chunk in reader:
    n_chunks += 1
    row_count += len(chunk)

# ---- 验收 ----
assert n_chunks == 3, f"块数应为 3，实际 {n_chunks}"
assert row_count == 270, f"总行数应为 270，实际 {row_count}"
print(f"TODO(6) 通过 -> {n_chunks} 块，共 {row_count} 行")

### TODO(7) 规范导出

把 `df_src` 的前 20 行导出一个 CSV 到 `output/answer_ch02.csv`，要求：

1. **不写索引列**
2. 编码用 **`utf-8-sig`**（Windows Excel 打开不乱码）
3. 缺失值统一写成 `NULL`

导完自己读回来验证一下。

In [ ]:
df_src = pd.read_csv(DATA / "device_defects.csv").head(20)
out_path = OUT_DIR / "answer_ch02.csv"

df_src.to_csv(out_path, index=False, encoding="utf-8-sig", na_rep="NULL")

# ---- 验收 ----
assert out_path.exists(), "文件没有生成"
raw_bytes = out_path.read_bytes()
assert raw_bytes[:3] == b"\xef\xbb\xbf", "缺少 UTF-8 BOM，编码不是 utf-8-sig"
assert b"Unnamed" not in raw_bytes, "写出了索引列"
back = pd.read_csv(out_path)
assert back.shape == (20, 13), f"读回来形状应为 (20, 13)，实际 {back.shape}"
text = out_path.read_text(encoding="utf-8-sig")
assert "NULL" in text, "缺失值没有被写成 NULL"
print("TODO(7) 通过 -> 已写出", out_path)

---

## 综合题：把「读进来 → 精简 → 导出去」走完整

目标是**生成一份干净的、可交付的 CSV**。请完成 3 个小步。

| 步骤 | 产物 | 说明 |
|---|---|---|
| ① 正确读入 | `src` | GBK 编码 + 编号列锁为字符串 |
| ② 读入时精简 | `slim` | 只要 4 列 |
| ③ 规范导出 | `output/final_ch02.csv` | 无索引、utf-8-sig、缺失写 NULL |

#### 综合 ① 正确读入（编码 + 类型）

In [ ]:
src = pd.read_csv(DATA / "device_defects_gbk.csv", encoding="gbk", dtype={"台区编号": "str", "设备编号": "str"})

# ---- 验收 ----
assert src.shape == (270, 13), f"形状应为 (270, 13)，实际 {src.shape}"
assert src["设备编号"].dtype == "str", "设备编号没有锁成字符串"
assert src["台区编号"].iloc[0].startswith("STATION_"), "编码不对导致内容乱码"
print("综合 ① 通过 ->", src.shape)

#### 综合 ② 读取阶段就精简列

In [ ]:
keep_cols = ["记录ID", "台区编号", "设备类型", "负荷值"]

slim = pd.read_csv(DATA / "device_defects_gbk.csv", encoding="gbk", usecols=keep_cols)

# ---- 验收 ----
assert list(slim.columns) == keep_cols, "列不对"
assert slim.shape == (270, 4), f"形状应为 (270, 4)，实际 {slim.shape}"
print("综合 ② 通过 ->", slim.shape)

#### 综合 ③ 规范导出并回读校验

In [ ]:
final_path = OUT_DIR / "final_ch02.csv"

slim.to_csv(final_path, index=False, encoding="utf-8-sig", na_rep="NULL")

# ---- 验收 ----
assert final_path.exists(), "文件没有生成"
assert final_path.read_bytes()[:3] == b"\xef\xbb\xbf", "编码不是 utf-8-sig"
round_trip = pd.read_csv(final_path)
assert round_trip.shape == (270, 4), f"回读形状应为 (270, 4)，实际 {round_trip.shape}"
assert list(round_trip.columns) == keep_cols, "回读列名不对，可能写出了索引列"
print("综合 ③ 通过 -> 交付文件:", final_path)

---

### 复盘提问

**竞赛复盘问自己**：如果这次数据换成 GBK 编码，我的脚本会在第几行崩？
把答案记进 `_notes/错题本.md`。